# BTC Orderbook LSTM — run.005 (EMA-cross entries + h8 fast-exit overlay)

**Why:** run.004 established that the h8 (2-min) head carries a real signal — pooled ensemble IC +0.0168, positive in all 4 folds, daily-IC t = +4.80 — but it is far below the fee floor **as an entry signal** (~$30–40 round trip vs. tiny per-trade edge). The user's EMA crossover on 5-min bars enters trends well but is blind to sudden collapses. run.005 tests the combination: **EMA cross opens the position, the h8 head force-exits it before a collapse.**

**Why the economics are different this time:** an exit-only overlay pays no extra fees when it is right (you were going to pay the exit side anyway) — its only cost is false alarms: each unnecessary exit + re-entry burns one round trip. That cost is measurable (re-entries × fees) and tunable (threshold k). And the exit doesn't need average-case accuracy; it needs **tail** sensitivity, which IC never measured. That is what this run measures first.

**Design (training cells are byte-frozen from run.004, including h240-based early stopping — so the h8 scores produced here are statistically the same ones that scored t = +4.8):**

1. **Event-study gate before any backtest** — for the worst 1% / 0.1% of forward 2-min (and 10-min) USD moves in the pooled walk-forward test bars: how often is the score in its bottom 1/5/10/20%? Lift vs. random, per fold. If the score is blind to the tail, the overlay is dead regardless of any backtest result.
2. **EMA grid on 5-min bars** — pairs (5/20, 9/21, 12/26, 20/50, 50/100, 50/200), EMA reset after data holes. **20/50 is the user's pre-registered primary; the rest of the grid is exploratory only** (six pairs × several thresholds = plenty of room to fool ourselves).
3. **Overlay state machine** — long-only primary (down-cross → flat), long/short variant reported. Fast exit when the score sits beyond −k·σ against the position for **EXIT_PERSIST=2 consecutive bars** (a one-bar spike is mostly noise, and on ~460k test bars even a 1% false-alarm rate would burn thousands of round trips; a real cascade dip persists for many bars — synthetic validation showed churn fees swamping crash savings without this filter). Hysteresis re-entry when the score recovers above −k·RESUME_FRAC after a cooldown; fresh entries vetoed while the score is beyond −k. MA legs pay maker (optimistic) in the "real" column; overlay legs always pay taker (they are urgent by construction); an all-taker column is the conservative bound.
4. **Random-exit control** — same machinery, score series circularly shifted (30 draws, primary pair): the true overlay must beat ~95% of controls on ΔmaxDD, otherwise "improvement" is just less time in market during a falling period.
5. **Scores persisted immediately** after training (npz) and copied to Drive at the end — run.004's scores died with the Colab VM.

**Decision rules (pre-registered):**
- **Gate (two-tier):** bottom-decile score capture of worst-1% 2-min drops with pooled lift ≥ 2× and ≥ 1.5× in every fold, **OR** of worst-0.1% drops with pooled lift ≥ 3× and ≥ 2× in every fold (the 1% tail contains much ordinary vol noise a true cascade detector can't and needn't capture; the 0.1% tail is where cascades dominate). Both tiers fail → stop; wait for schema-4 data.
- **Overlay wins** if, for the primary 20/50 pair: maxDD cut ≥ 20% AND net P&L (real fees) ≥ MA-only baseline AND beats ≥ 95% of random controls on ΔmaxDD, with neighbouring k values agreeing in direction.
- Everything else in the grid is hypothesis-generating, not evidence.

**Caveats:** no slippage model — taker fills *during a cascade* are worse than bar-close prints, so overlay results are an upper bound; collector gaps >1h force both arms flat (fee churn hits baseline and overlay equally); backtest trades only on valid-sample bars (~85% of test bars), holding across small gaps ≤1h.

Expected runtime on a T4: **roughly 2–4 h** (training identical to run.004; XGB cell dropped; overlay cells add minutes).

## Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order


In [ ]:
# ── Cell 1: Install and imports ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'scipy',
                'xgboost'], check=True)

import os, glob, gc, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: (optional) data transfer helpers — uncomment what you need ────
#!apt install sshpass -y && sshpass -p XXXXXXXXXX sftp -r -o "StrictHostKeyChecking no" scpuser@155.207.120.2:btc_data.tar.xz .
#from google.colab import drive
#drive.mount('/content/drive')
#!mv btc_data.tar.xz /content/drive/MyDrive/
#!ls /content/drive/MyDrive/ -l
#drive.flush_and_unmount()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cd ~ && mkdir -p btc_lstm_v2
!cd ~ && tar xJf /content/drive/MyDrive/btc_data.tar.xz

drive.flush_and_unmount()

In [ ]:
# ── Cell 3: Mount Google Drive, extract data, config ─────────────────────


# ── CONFIGURE THESE ───────────────────────────────────────────────────────
DATA_DIR   = '/root/btc_data'
OUTPUT_DIR = '/root/btc_lstm_run005'

WINDOW_SEC  = 15          # bar size produced by the collector
HORIZONS    = [8, 40, 240]  # frozen from run.004 (h240 head kept so training
H_TRADE     = 240           # and early stopping are byte-identical)
VOL_WINDOW  = 240         # 1h rolling window for target normalisation
TARGET_CLIP = 5.0         # clip |target| at 5 sigma

SEQ_LEN      = 64         # 16 min of context
HIDDEN_DIM   = 128        # SMALL on purpose — run.001 overfit already at 212K params
NUM_LAYERS   = 2
DROPOUT      = 0.3
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 2048
EPOCHS       = 12
WARMUP       = 2          # linear warmup epochs before cosine decay
PATIENCE     = 5          # early stop on val rank-IC of the trading head
EPOCH_SAMPLE_CAP = 600_000  # train sequences drawn per epoch (weighted, w/ replacement)

# ── run.003 (kept): focus training on directional bars ──
SAMPLE_W_BASE = 0.25      # epoch-draw probability ∝ SAMPLE_W_BASE + |y_trade|
LOSS_W_ALPHA  = 1.0       # per-sample Huber weight = min(1 + α·|y_trade|, cap)
LOSS_W_CAP    = 4.0

# ── run.004 (kept): multi-seed evaluation ──
SEEDS = [0, 1, 2]         # per-fold; test score = seed-ensemble mean
MIN_DAY_SAMPLES = 500     # a test day needs this many samples to enter daily-IC stats

N_FOLDS         = 4       # walk-forward folds over the last 60% of the data
TEST_START_FRAC = 0.40    # first 40% is never tested (training warmup)
VAL_FRAC        = 0.12    # tail of each training range used for early stopping
COVERAGE_MIN    = 0.60    # features with lower non-NaN coverage are excluded

MAKER_FEE = 0.0002        # per side (Binance USDT-M futures maker)
TAKER_FEE = 0.0005        # per side

# ── run.005: EMA-cross entries + fast-exit overlay ──
EXIT_H     = 8            # exit-signal head: 2-min horizon (t=+4.8 in run.004)
EXIT_H_ALT = 40           # alternative exit head to compare (10-min, t=+2.2)
EMA_BAR    = '5min'       # the user's validated bar size
EMA_PAIRS  = [(5, 20), (9, 21), (12, 26), (20, 50), (50, 100), (50, 200)]
EMA_PRIMARY = (20, 50)    # pre-registered primary; other pairs are exploratory
EXIT_KS    = [1.5, 2.0, 2.5, 3.0, 4.0]  # fast-exit trigger, in val-σ score units
EXIT_PERSIST = 2          # consecutive bars beyond −k required to exit: one-bar
                          # score spikes are mostly noise and each false alarm
                          # burns a full round trip; a real cascade dip persists
RESUME_FRAC = 0.25        # re-enter when score recovers above −k·RESUME_FRAC
REENTRY_COOLDOWN = 8      # bars (2 min) minimum flat time after a fast exit
ENTRY_VETO = True         # veto fresh MA entries while score is beyond −k
HOLD_GAP_MAX  = 3600      # positions may be held across data gaps up to this (sec)
EMA_STALE_MAX = 600       # no MA signal if the last 5-min bar is older than this (sec)
N_CONTROL     = 30        # circular-shift random-exit controls (primary pair only)
DRIVE_SAVE_DIR = '/content/drive/MyDrive/btc_lstm_run005'
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data dir:   {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
# ── Cell 4: Feature pipeline ──────────────────────────────────────────────
SEPARATOR = ';'

COLS_NEEDED_BASE = {
    'spot_datetime', 'future_datetime', 'spot_timestamp', 'future_timestamp',
    'future_bid_close', 'future_ask_close',
    'future_bid_open', 'future_bid_max', 'future_bid_min',
    'future_bid_median', 'future_ask_median',
    'future_spread_open', 'future_spread_max',
    'future_buy_qty', 'future_sell_qty',
    'future_buy_samples', 'future_sell_samples',
    'future_buy_vwap', 'future_sell_vwap', 'future_price_diff',
    'future_bid_liq_0.0_median', 'future_ask_liq_0.0_median',
    'spot_buy_qty', 'spot_sell_qty',
    'spot_bid_close', 'spot_ask_close',
    'opt_open_interest_sample', 'opt_funding_rate_sample',
    'opt_est_funding_rate_sample', 'opt_remaining_time_sample',
    'opt_long_force_exit_qty_sum', 'opt_short_force_exit_qty_sum',
    'future_first_trade_side', 'future_last_trade_side',
    'future_largest_trade_qty', 'future_largest_trade_side',
    'future_buy_count_early', 'future_buy_count_late',
    'future_sell_count_early', 'future_sell_count_late',
    'future_buy_qty_early', 'future_buy_qty_late',
    'future_sell_qty_early', 'future_sell_qty_late',
}
COLS_NEEDED_LIQ_DEEP  = [
    'future_bid_liq_0.04_median', 'future_ask_liq_0.04_median',
    'future_bid_liq_0.05_median', 'future_ask_liq_0.05_median',
    'future_bid_liq_0.06_median', 'future_ask_liq_0.06_median',
]
COLS_NEEDED_LIQ_TOTAL = [
    'future_bid_liq_0.1_median',  'future_ask_liq_0.1_median',
    'future_bid_liq_0.2_median',  'future_ask_liq_0.2_median',
    'future_bid_liq_0.3_median',  'future_ask_liq_0.3_median',
    'future_bid_liq_0.4_median',  'future_ask_liq_0.4_median',
]

# run.004: long-timescale context the 64-bar input window cannot see
NEW_FEATURES = [
    'ret_norm_1h', 'ret_norm_4h', 'ret_norm_24h',
    'ma_gap_1h', 'ma_gap_4h', 'ma_gap_24h',
    'vol_ratio_1h_24h', 'range_pos_24h',
    'basis_z_4h', 'basis_mom_1h',
    'dow_sin', 'dow_cos',
]

ALL_FEATURES = [
    'book_imbalance', 'flow_imbalance', 'momentum', 'book_imbal_deep',
    'flow_imbal_roll4', 'flow_imbal_roll8', 'book_imbal_roll4', 'book_imbal_roll8',
    'vwap_spread', 'liq_flag', 'stochastic', 'spread_expansion', 'sample_imbalance',
    'flow_agreement', 'oi_change', 'size_imbalance', 'liq_conc_bid', 'liq_conc_ask',
    'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'near_funding', 'funding_pressure', 'vol_norm',
    'trade_side_open', 'trade_side_close', 'trade_side_momentum',
    'largest_trade_side', 'largest_trade_rel',
    'buy_accel', 'sell_accel', 'flow_accel', 'buy_count_accel', 'late_imbalance',
] + NEW_FEATURES


def load_files(files):
    all_needed = COLS_NEEDED_BASE | set(COLS_NEEDED_LIQ_DEEP) | set(COLS_NEEDED_LIQ_TOTAL)
    frames = []
    for path in files:
        try:
            hdr = pd.read_csv(path, sep=SEPARATOR, nrows=0)
            usecols = sorted(all_needed & set(hdr.columns))
            if 'spot_datetime' not in usecols:
                print(f'  SKIP {os.path.basename(path)}: missing spot_datetime')
                continue
            df = pd.read_csv(path, sep=SEPARATOR, usecols=usecols, low_memory=False)
            df = df[df['spot_datetime'] != 'spot_datetime']
            frames.append(df)
        except Exception as e:
            print(f'  SKIP {os.path.basename(path)}: {e}')
    if not frames:
        raise RuntimeError('No valid CSV files found.')
    print(f'  Loaded {len(frames)} files')
    return pd.concat(frames, ignore_index=True)


def prepare(df, horizons, vol_window):
    df['dt'] = pd.to_datetime(df['spot_datetime'], errors='coerce')
    df = df.dropna(subset=['dt']).sort_values('dt').reset_index(drop=True)
    n_before = len(df)
    df = df.drop_duplicates(subset='dt', keep='first').reset_index(drop=True)
    if len(df) < n_before:
        print(f'  Dropped {n_before - len(df)} duplicate timestamps')

    skip = {'spot_datetime', 'future_datetime', 'dt'}
    for col in df.columns:
        if col not in skip:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in [c for c in df.columns if c.startswith('opt_')]:
        df[col] = df[col].replace(-1, np.nan)

    if 'future_bid_close' in df.columns and 'future_ask_close' in df.columns:
        df['close_price'] = (df['future_bid_close'] + df['future_ask_close']) / 2.0
    else:
        df['close_price'] = (df['future_bid_median'] + df['future_ask_median']) / 2.0

    # Gap structure: collector restarts / dropped bars break the 15s cadence.
    gap = df['dt'].diff().dt.total_seconds()
    n_breaks = int((gap != WINDOW_SEC).sum()) - 1
    big = gap[gap > 3600]
    print(f'  Cadence breaks: {n_breaks}  (gaps >1h: {len(big)}, largest: {gap.max()/3600:.1f}h)')

    eps = 1e-9
    df['flow_imbalance'] = ((df['future_buy_qty'] - df['future_sell_qty']) /
                            (df['future_buy_qty'] + df['future_sell_qty'] + eps))
    bid0 = df['future_bid_liq_0.0_median']
    ask0 = df['future_ask_liq_0.0_median']
    df['book_imbalance'] = (bid0 - ask0) / (bid0 + ask0 + eps)
    df['momentum'] = df['future_price_diff']

    deep_col = None
    for lvl in ['0.05', '0.04', '0.06']:
        b, a = f'future_bid_liq_{lvl}_median', f'future_ask_liq_{lvl}_median'
        if b in df.columns and a in df.columns:
            deep_col = (b, a); break
    df['book_imbal_deep'] = ((df[deep_col[0]] - df[deep_col[1]]) /
                             (df[deep_col[0]] + df[deep_col[1]] + eps)) if deep_col else np.nan

    df['flow_imbal_roll4'] = df['flow_imbalance'].rolling(4, min_periods=2).mean()
    df['flow_imbal_roll8'] = df['flow_imbalance'].rolling(8, min_periods=4).mean()
    df['book_imbal_roll4'] = df['book_imbalance'].rolling(4, min_periods=2).mean()
    df['book_imbal_roll8'] = df['book_imbalance'].rolling(8, min_periods=4).mean()

    df['vwap_spread'] = (df['future_buy_vwap'] - df['future_sell_vwap']
                         if 'future_buy_vwap' in df.columns else np.nan)
    liq_long  = df.get('opt_long_force_exit_qty_sum',  pd.Series(0, index=df.index))
    liq_short = df.get('opt_short_force_exit_qty_sum', pd.Series(0, index=df.index))
    df['liq_flag'] = ((liq_long + liq_short) > 0).astype(float)

    if all(c in df.columns for c in ['future_bid_max', 'future_bid_min', 'future_bid_close']):
        rng = df['future_bid_max'] - df['future_bid_min']
        df['stochastic'] = np.where(rng > 0, (df['future_bid_close'] - df['future_bid_min']) / rng, 0.5)
    else:
        df['stochastic'] = np.nan

    df['spread_expansion'] = (df['future_spread_max'] - df['future_spread_open']
                              if all(c in df.columns for c in ['future_spread_max', 'future_spread_open'])
                              else np.nan)

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples']):
        bs, ss = df['future_buy_samples'], df['future_sell_samples']
        df['sample_imbalance'] = (bs - ss) / (bs + ss + eps)
    else:
        df['sample_imbalance'] = np.nan

    if all(c in df.columns for c in ['spot_buy_qty', 'spot_sell_qty']):
        spot_flow = ((df['spot_buy_qty'] - df['spot_sell_qty']) /
                     (df['spot_buy_qty'] + df['spot_sell_qty'] + eps))
        df['flow_agreement'] = df['flow_imbalance'] * spot_flow
    else:
        df['flow_agreement'] = np.nan

    df['oi_change'] = df['opt_open_interest_sample'].diff() if 'opt_open_interest_sample' in df.columns else np.nan

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples',
                                      'future_buy_qty', 'future_sell_qty']):
        df['size_imbalance'] = (df['future_buy_qty']  / (df['future_buy_samples']  + eps) -
                                df['future_sell_qty'] / (df['future_sell_samples'] + eps))
    else:
        df['size_imbalance'] = np.nan

    sub_cols = ['future_first_trade_side', 'future_last_trade_side',
                'future_largest_trade_qty', 'future_largest_trade_side',
                'future_buy_count_early', 'future_buy_count_late',
                'future_sell_count_early', 'future_sell_count_late',
                'future_buy_qty_early', 'future_buy_qty_late',
                'future_sell_qty_early', 'future_sell_qty_late']
    if all(c in df.columns for c in sub_cols):
        bqe, bql = df['future_buy_qty_early'],  df['future_buy_qty_late']
        sqe, sql = df['future_sell_qty_early'],  df['future_sell_qty_late']
        fts, lts = df['future_first_trade_side'], df['future_last_trade_side']
        lq,  ls  = df['future_largest_trade_qty'], df['future_largest_trade_side']
        bce, bcl = df['future_buy_count_early'], df['future_buy_count_late']
        total_vol = bqe + bql + sqe + sql
        df['trade_side_open']     = fts
        df['trade_side_close']    = lts
        df['trade_side_momentum'] = lts - fts
        df['largest_trade_side']  = ls
        df['largest_trade_rel']   = lq / (total_vol + eps)
        df['buy_accel']           = bql - bqe
        df['sell_accel']          = sql - sqe
        df['flow_accel']          = (bql - sql) - (bqe - sqe)
        df['buy_count_accel']     = bcl - bce
        df['late_imbalance']      = (bql - sql) / (bql + sql + eps)
    else:
        for c in ['trade_side_open','trade_side_close','trade_side_momentum',
                  'largest_trade_side','largest_trade_rel','buy_accel','sell_accel',
                  'flow_accel','buy_count_accel','late_imbalance']:
            df[c] = np.nan

    deep_bid = next((c for c in ['future_bid_liq_0.4_median','future_bid_liq_0.3_median',
                                  'future_bid_liq_0.2_median','future_bid_liq_0.1_median']
                     if c in df.columns), None)
    deep_ask = next((c for c in ['future_ask_liq_0.4_median','future_ask_liq_0.3_median',
                                  'future_ask_liq_0.2_median','future_ask_liq_0.1_median']
                     if c in df.columns), None)
    df['liq_conc_bid'] = (df['future_bid_liq_0.0_median'] / (df[deep_bid] + eps)
                          if deep_bid else np.nan)
    df['liq_conc_ask'] = (df['future_ask_liq_0.0_median'] / (df[deep_ask] + eps)
                          if deep_ask else np.nan)

    hour, minute = df['dt'].dt.hour, df['dt'].dt.minute
    df['hour_sin']   = np.sin(2 * np.pi * hour   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * hour   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * minute / 60)
    df['minute_cos'] = np.cos(2 * np.pi * minute / 60)

    secs = (hour * 3600 + minute * 60 + df['dt'].dt.second) % (8 * 3600)
    df['near_funding'] = ((8 * 3600 - secs) < 900).astype(float)
    if 'opt_funding_rate_sample' in df.columns:
        funding = df['opt_funding_rate_sample'].replace(-1, np.nan).fillna(0)
        df['funding_pressure'] = funding * df['near_funding']
    else:
        df['funding_pressure'] = np.nan

    df['volatility'] = df['close_price'].diff().abs().rolling(16, min_periods=4).mean()
    df['vol_norm']   = (df['volatility'] /
                        df['volatility'].rolling(5760, min_periods=96).mean()
                       ).replace([np.inf, -np.inf], np.nan)

    # ── run.004: long-timescale context features ─────────────────────────
    # All rolling windows are TIME-based on the datetime index (rolling('24h')),
    # not row-count based: collector restarts land on arbitrary second offsets
    # and gaps would otherwise stretch a "1h" window over more wall time.
    # Everything is causal (past data only) and vol-normalised so the scale is
    # stationary across regimes; clipped at ±8 to bound scaler-era outliers.
    D = 24 * 3600 // WINDOW_SEC              # bars per 24h at full cadence
    lp = pd.Series(np.log(df['close_price'].values), index=df['dt'])

    def past(series, delta, tol_sec=2 * WINDOW_SEC):
        # value of `series` ~delta earlier (nearest bar within tol), aligned back
        p = series.reindex(series.index - delta, method='nearest',
                           tolerance=pd.Timedelta(seconds=tol_sec))
        return pd.Series(p.values, index=series.index)

    r1 = lp.diff()
    r1[gap.values != WINDOW_SEC] = np.nan    # 1-bar return across a gap is not 1-bar
    sigma24 = r1.rolling('24h', min_periods=D // 4).std()

    for name, td, W in [('1h', pd.Timedelta(hours=1),  D // 24),
                        ('4h', pd.Timedelta(hours=4),  D // 6),
                        ('24h', pd.Timedelta(hours=24), D)]:
        unit = sigma24 * np.sqrt(W) + eps
        df[f'ret_norm_{name}'] = ((lp - past(lp, td)) / unit).clip(-8, 8).values
        ma = lp.rolling(td, min_periods=int(W * 0.75)).mean()
        df[f'ma_gap_{name}'] = ((lp - ma) / unit).clip(-8, 8).values

    vol1h = r1.rolling('1h', min_periods=D // 48).std()
    df['vol_ratio_1h_24h'] = (vol1h / (sigma24 + eps)).clip(0, 5).values

    hi24 = lp.rolling('24h', min_periods=D // 4).max()
    lo24 = lp.rolling('24h', min_periods=D // 4).min()
    df['range_pos_24h'] = ((lp - lo24) / (hi24 - lo24 + eps)).clip(0, 1).values

    if all(c in df.columns for c in ['spot_bid_close', 'spot_ask_close']):
        spot_mid = (df['spot_bid_close'] + df['spot_ask_close']) / 2.0
        basis = pd.Series(((df['close_price'] - spot_mid) / (spot_mid + eps)).values,
                          index=df['dt'])
        b_ma = basis.rolling('4h', min_periods=D // 12).mean()
        b_sd = basis.rolling('4h', min_periods=D // 12).std()
        df['basis_z_4h'] = ((basis - b_ma) / (b_sd + eps)).clip(-8, 8).values
        b_mom = basis - past(basis, pd.Timedelta(hours=1))
        b_mom_sd = b_mom.rolling('4h', min_periods=D // 12).std()
        df['basis_mom_1h'] = (b_mom / (b_mom_sd + eps)).clip(-8, 8).values
    else:
        df['basis_z_4h'] = np.nan
        df['basis_mom_1h'] = np.nan

    dow = df['dt'].dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    del lp, r1, sigma24, vol1h, hi24, lo24
    # ──────────────────────────────────────────────────────────────────────

    # Targets: volatility-normalised future return per horizon. A target is
    # valid only when the row h bars ahead really is h*15s ahead — no labels
    # across collector gaps. vol uses only past prices (no leakage).
    # (datetime64[s] cast is robust to the pandas 2/3 resolution change)
    dt_sec = pd.Series(df['dt'].values.astype('datetime64[s]').astype(np.int64),
                       index=df.index)
    for h in horizons:
        vol   = df['close_price'].diff(h).rolling(vol_window, min_periods=vol_window // 4).std()
        delta = df['close_price'].shift(-h) - df['close_price']
        y     = (delta / (vol + eps)).clip(-TARGET_CLIP, TARGET_CLIP)
        valid = (dt_sec.shift(-h) - dt_sec) == h * WINDOW_SEC
        y[~valid.fillna(False)] = np.nan
        df[f'y_{h}'] = y
    return df

print('Feature pipeline loaded.')

In [ ]:
# ── Cell 5: Load data, coverage guard, build arrays ───────────────────────
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')) +
               glob.glob(os.path.join(DATA_DIR, '*.csv.gz')))
print(f'Found {len(files)} files')

raw = load_files(files)
print(f'Total rows loaded: {len(raw):,}')

df = prepare(raw, HORIZONS, VOL_WINDOW)
del raw; gc.collect()

# Coverage guard: a feature that exists only in the newer collector files
# (e.g. sub-bar features present only in ~15 days of v2 data) would otherwise
# make dropna() silently discard the entire older year of data.
features = []
for f_ in ALL_FEATURES:
    if f_ not in df.columns:
        continue
    cov = df[f_].notna().mean()
    if cov >= COVERAGE_MIN:
        features.append(f_)
    else:
        print(f'  EXCLUDED {f_:22s} coverage {cov*100:5.1f}% < {COVERAGE_MIN*100:.0f}%')
print(f'Features kept: {len(features)}/{len(ALL_FEATURES)}')

n0 = len(df)
clean = df[features + [f'y_{h}' for h in HORIZONS] + ['dt', 'close_price']]           .dropna(subset=features).reset_index(drop=True)
lost = n0 - len(clean)
print(f'Rows: {n0:,} → {len(clean):,} after dropna ({lost/max(n0,1)*100:.2f}% dropped)')
if lost > 0.05 * n0:
    print('\n*** WARNING: dropna removed >5% of rows — check per-feature NaN fractions: ***')
    print(df[features].isna().mean().sort_values(ascending=False).head(10))
del df; gc.collect()

X_raw  = clean[features].values.astype(np.float32)
Y_all  = np.stack([clean[f'y_{h}'].values for h in HORIZONS], axis=1).astype(np.float32)
dt_s   = clean['dt'].values.astype('datetime64[s]').astype(np.int64)
dt_all = clean['dt'].values
close  = clean['close_price'].values.astype(np.float64)
n, F_DIM = X_raw.shape
del clean; gc.collect()

# A sample "ends" at row i: input window = rows [i-SEQ_LEN+1, i], targets at
# i+h. Valid only if the window is contiguous 15s bars (dropna and collector
# gaps both break cadence, and the dt check catches both) and targets exist.
contig = np.zeros(n, dtype=bool)
span = (SEQ_LEN - 1) * WINDOW_SEC
contig[SEQ_LEN-1:] = (dt_s[SEQ_LEN-1:] - dt_s[:n-SEQ_LEN+1]) == span
sample_valid = contig & np.isfinite(Y_all).all(axis=1)
assert sample_valid.sum() > 0, 'No valid samples — check bar cadence / targets'

print(f'\nRows: {n:,}   valid samples: {sample_valid.sum():,} ({sample_valid.mean()*100:.1f}%)')
print(f'Date range: {pd.Timestamp(dt_all[0])} → {pd.Timestamp(dt_all[-1])}')
H_TRADE_IDX = HORIZONS.index(H_TRADE)
yt = Y_all[sample_valid, H_TRADE_IDX]
print(f'Trading target y_{H_TRADE}: std={yt.std():.3f}   |y|>1σ: {(np.abs(yt) > 1).mean()*100:.1f}%   '
      f'up share: {(yt > 0).mean()*100:.1f}%')

In [ ]:
# ── Cell 6: Walk-forward folds (expanding window, purged) ─────────────────
MAX_H = max(HORIZONS)

def ends_in(a, b):
    return np.flatnonzero(sample_valid[a:b]) + a

test_start = int(n * TEST_START_FRAC)
chunk = (n - test_start) // N_FOLDS
folds = []
for k in range(N_FOLDS):
    lo = test_start + k * chunk
    hi = n if k == N_FOLDS - 1 else lo + chunk
    boundary = lo - MAX_H                     # purge: val labels must not reach into test
    val_lo   = int(boundary * (1 - VAL_FRAC))
    folds.append({'k': k,
                  'train_hi': val_lo - MAX_H,  # purge: train labels must not reach into val
                  'val':  (val_lo, boundary),
                  'test': (lo, hi)})

print('Fold   n_train     n_val    n_test    test period')
for f_ in folds:
    t_lo, t_hi = f_['test']
    n_tr = len(ends_in(0, f_['train_hi']))
    n_va = len(ends_in(*f_['val']))
    n_te = len(ends_in(t_lo, t_hi))
    assert min(n_tr, n_va, n_te) > 0, f"fold {f_['k']} has an empty split"
    print(f"  {f_['k']}  {n_tr:>9,}  {n_va:>8,}  {n_te:>8,}    "
          f"{pd.Timestamp(dt_all[t_lo])} → {pd.Timestamp(dt_all[t_hi-1])}")

In [ ]:
# ── Cell 7: Model — small LSTM, multi-horizon regression heads ────────────
class BtcLSTMReg(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, n_out):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, n_out))

    def forward(self, x):
        h, _ = self.lstm(self.input_norm(x))
        return self.head(h[:, -1])

_tmp = BtcLSTMReg(F_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, len(HORIZONS))
print(f'Model parameters: {sum(p.numel() for p in _tmp.parameters()):,}')
del _tmp

In [ ]:
# ── Cell 8: Training helpers (zero-copy window gathering, multi-seed) ─────
AMP = device.type == 'cuda'

def make_scheduler(opt, warmup, total):
    def fn(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        prog = (ep - warmup) / max(total - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def ic(a, b):
    v = spearmanr(a, b)[0]
    return float(v) if np.isfinite(v) else 0.0

def predict(model, view, ends, batch=4096):
    model.eval()
    outs = []
    with torch.no_grad():
        for s in range(0, len(ends), batch):
            e = ends[s:s+batch]
            xb = torch.from_numpy(view[e - (SEQ_LEN - 1)]).to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                outs.append(model(xb).float().cpu())
    return torch.cat(outs).numpy()

def train_one_seed(seed, view, train_ends, val_ends, samp_p):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    model = BtcLSTMReg(F_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, len(HORIZONS)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(opt, WARMUP, EPOCHS)
    scaler_amp = torch.amp.GradScaler('cuda', enabled=AMP)

    best_ic, best_state, best_epoch, wait = -9.0, None, 0, 0
    hist = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        # weighted draw WITH replacement (weighted w/o replacement is O(k·n)
        # in numpy); duplicates are just oversampling, which is the point
        ep_ends = rng.choice(train_ends,
                             min(len(train_ends), EPOCH_SAMPLE_CAP),
                             replace=True, p=samp_p)
        tot, m = 0.0, 0
        for s in range(0, len(ep_ends), BATCH_SIZE):
            e = ep_ends[s:s+BATCH_SIZE]
            xb = torch.from_numpy(view[e - (SEQ_LEN - 1)]).to(device, non_blocking=True)
            yb = torch.from_numpy(Y_all[e]).to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                per = F.smooth_l1_loss(model(xb), yb, reduction='none').mean(dim=1)
                w = torch.clamp(1.0 + LOSS_W_ALPHA * yb[:, H_TRADE_IDX].abs(),
                                max=LOSS_W_CAP)
                loss = (w * per).sum() / w.sum()
            opt.zero_grad(set_to_none=True)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(opt)
            scaler_amp.update()
            tot += loss.item() * len(e); m += len(e)
        sched.step()

        vp = predict(model, view, val_ends)
        v_ic = [ic(vp[:, i], Y_all[val_ends, i]) for i in range(len(HORIZONS))]
        hist.append({'train_loss': tot / m, 'val_ic': v_ic})
        cur = v_ic[H_TRADE_IDX]
        if cur > best_ic:
            best_ic, best_epoch, wait = cur, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            flag = '  ← best'
        else:
            wait += 1
            flag = f'  ({wait}/{PATIENCE})'
        ics = '  '.join(f'IC_h{h}={v:+.4f}' for h, v in zip(HORIZONS, v_ic))
        print(f'    seed {seed}  ep {epoch:2d}  loss={tot/m:.4f}  {ics}{flag}')
        if wait >= PATIENCE:
            print('    early stop')
            break

    model.load_state_dict(best_state)
    return model, best_state, best_epoch, hist

def train_fold(fold):
    t_hi = fold['train_hi']
    v0, v1 = fold['val']
    s0, s1 = fold['test']

    scaler = StandardScaler().fit(X_raw[:t_hi])
    Xs = scaler.transform(X_raw).astype(np.float32)
    # (n-SEQ_LEN+1, SEQ_LEN, F) VIEW into Xs — no copy; batches materialise
    # only when fancy-indexed below. This is what keeps RAM under control.
    view = np.lib.stride_tricks.sliding_window_view(Xs, (SEQ_LEN, F_DIM))[:, 0]

    train_ends = ends_in(0, t_hi)
    val_ends   = ends_in(v0, v1)
    test_ends  = ends_in(s0, s1)

    # Importance sampling: bars with bigger vol-normalised moves are drawn
    # more often; SAMPLE_W_BASE keeps flat bars in the mix so the model still
    # learns what "nothing happening" looks like.
    samp_w = SAMPLE_W_BASE + np.abs(Y_all[train_ends, H_TRADE_IDX]).astype(np.float64)
    samp_p = samp_w / samp_w.sum()
    eff = 1.0 / np.sum(samp_p ** 2) / len(samp_p)
    print(f'  sampling tilt: effective sample fraction {eff*100:.1f}% '
          f'(100% = uniform)')

    # run.004: identical training per seed; only the RNG differs. The test
    # score is the seed-ensemble mean; per-seed ICs expose how much of any
    # single-seed result is initialisation luck.
    states, best_epochs, hists = [], [], []
    val_preds, test_preds, seed_test_ic = [], [], []
    for seed in SEEDS:
        model, best_state, best_epoch, hist = train_one_seed(
            seed, view, train_ends, val_ends, samp_p)
        vp = predict(model, view, val_ends)
        tp = predict(model, view, test_ends)
        s_ic = [ic(tp[:, i], Y_all[test_ends, i]) for i in range(len(HORIZONS))]
        ics = '  '.join(f'IC_h{h}={v:+.4f}' for h, v in zip(HORIZONS, s_ic))
        print(f'    seed {seed} TEST: {ics}   (best epoch {best_epoch})')
        states.append(best_state); best_epochs.append(best_epoch); hists.append(hist)
        val_preds.append(vp); test_preds.append(tp); seed_test_ic.append(s_ic)
        del model

    val_pred  = np.mean(val_preds, axis=0)     # seed-ensemble
    test_pred = np.mean(test_preds, axis=0)
    test_ic = [ic(test_pred[:, i], Y_all[test_ends, i]) for i in range(len(HORIZONS))]
    out = {'scaler': scaler, 'states': states, 'best_epochs': best_epochs,
           'hist': hists[-1], 'val_ends': val_ends, 'val_pred': val_pred,
           'test_ends': test_ends, 'test_pred': test_pred,
           'test_ic': test_ic, 'seed_test_ic': seed_test_ic}
    del Xs, view
    gc.collect()
    return out

In [ ]:
# ── Cell 9: Run walk-forward ──────────────────────────────────────────────
results = []
for fold in folds:
    lo, hi = fold['test']
    print(f"\n════ Fold {fold['k']}: test {pd.Timestamp(dt_all[lo])} → {pd.Timestamp(dt_all[hi-1])} ════")
    res = train_fold(fold)
    results.append(res)
    ics = '  '.join(f'IC_h{h}={v:+.4f}' for h, v in zip(HORIZONS, res['test_ic']))
    print(f"  ENSEMBLE TEST ({len(res['test_ends']):,} samples): {ics}")

print('\n──── Walk-forward summary (seed-ensemble; ± = per-seed spread) ────')
hdr = '  '.join(f"{'IC_h' + str(h):>16s}" for h in HORIZONS)
print(f'Fold    n_test   {hdr}')
for f_, r in zip(folds, results):
    cells = []
    for j in range(len(HORIZONS)):
        s_ics = [r['seed_test_ic'][s][j] for s in range(len(SEEDS))]
        cells.append(f"{r['test_ic'][j]:>+8.4f} ±{np.std(s_ics):.4f}")
    print(f"  {f_['k']}   {len(r['test_ends']):>8,}  " + '  '.join(cells))

ends_pool = np.concatenate([r['test_ends'] for r in results])
pred_pool = np.concatenate([r['test_pred'] for r in results])
pooled_ic = [ic(pred_pool[:, i], Y_all[ends_pool, i]) for i in range(len(HORIZONS))]
print('Pooled ensemble:   ' + '  '.join(f'{v:>+9.4f}       ' for v in pooled_ic))

# Backtest/pooling score: per-fold predictions divided by that fold's
# VALIDATION score std, so "σ" entry thresholds mean the same thing in every
# fold and the score scale is chosen without touching test data.
score_pool = np.concatenate([r['test_pred'] / (r['val_pred'].std(axis=0) + 1e-9)
                             for r in results])

# Daily IC: one IC per test day, then a t-stat across days. Labels overlap
# heavily WITHIN a day, but day-level ICs are close to independent draws —
# this is the noise-aware number to compare between runs.
def daily_ic_stats(ends, score, y):
    days = dt_all[ends].astype('datetime64[D]')
    vals = []
    for d in np.unique(days):
        m = days == d
        if m.sum() >= MIN_DAY_SAMPLES:
            vals.append(ic(score[m], y[m]))
    vals = np.array(vals)
    se = vals.std(ddof=1) / np.sqrt(len(vals))
    return vals.mean(), se, vals.mean() / se, len(vals)

print('\n──── Daily-IC t-stats (pooled ensemble; |t|>2 ≈ significant) ────')
for j, h in enumerate(HORIZONS):
    mu, se, t, k = daily_ic_stats(ends_pool, score_pool[:, j], Y_all[ends_pool, j])
    print(f'  h{h:<4d}  mean daily IC {mu:+.4f} ± {se:.4f}   t = {t:+.2f}   ({k} days)')

print('\nNote: per-fold consistency and the daily-IC t-stat matter more than any')
print('single pooled number. Compare h8/h40 against run.003 (pure feature effect);')
print('h240 is the new hypothesis. Sign flips across folds = no stable edge.')

# ── run.005: persist scores IMMEDIATELY — run.004's scores died with the VM ──
fold_of = np.concatenate([np.full(len(r['test_ends']), f_['k'])
                          for f_, r in zip(folds, results)])
scores_path = os.path.join(OUTPUT_DIR, 'run005_scores.npz')
np.savez_compressed(scores_path,
    ends=ends_pool, score=score_pool, horizons=np.array(HORIZONS),
    dt_s=dt_s[ends_pool], close=close[ends_pool],
    y=Y_all[ends_pool], fold_of=fold_of)
print(f'\nScores persisted → {scores_path}')
print('(copied to Drive in the final cell — if you plan to stop early, copy it now)')


In [ ]:
# ── Cell 10: Event study — does the h8 score fire before collapses? ───────
# The overlay premise is that the 2-min head goes strongly negative BEFORE a
# collapse. IC measures average rank correlation over all bars; this cell
# measures tail behaviour specifically: of the worst q% forward moves, what
# fraction had a score in its bottom qs% AT the bar where the move starts
# (exiting at that bar's close avoids the whole move)? Lift = capture / qs;
# 1.0x = the score is blind to the tail. Per fold first — a tail signal that
# exists in only one regime is not a tail signal.
#
# This is the GATE for everything below: if it fails, the overlay backtest
# results are noise-mining and must not be believed.

EXIT_H_IDX     = HORIZONS.index(EXIT_H)
EXIT_H_ALT_IDX = HORIZONS.index(EXIT_H_ALT)
s8  = score_pool[:, EXIT_H_IDX]
s40 = score_pool[:, EXIT_H_ALT_IDX]
# raw USD forward moves (crashes are defined in USD — that is what a held
# position loses; sample validity guarantees rows i+8 / i+40 are contiguous)
fwd8  = close[ends_pool + EXIT_H]     - close[ends_pool]
fwd40 = close[ends_pool + EXIT_H_ALT] - close[ends_pool]

SCORE_QS = (0.01, 0.05, 0.10, 0.20)

def capture_table(score, fwd, crash_q, title):
    print(f'\n──── {title}: worst {crash_q*100:g}% forward moves ────')
    hdr = '  '.join(f'score bottom {q*100:>2.0f}%' for q in SCORE_QS)
    print(f'  {"scope":8s}  {"crash ≤":>9s}  {hdr}   (lift, capture)')
    scopes = [(f'fold {k}', fold_of == k) for k in range(N_FOLDS)] + \
             [('POOLED', np.ones(len(score), dtype=bool))]
    pooled_lifts = None
    for name, m in scopes:
        s_, f_ = score[m], fwd[m]
        cthr  = np.quantile(f_, crash_q)
        crash = f_ <= cthr
        cells, lifts = [], []
        for qs in SCORE_QS:
            cap  = (s_[crash] <= np.quantile(s_, qs)).mean()
            lifts.append(cap / qs)
            cells.append(f'{cap/qs:>5.2f}x {cap*100:>5.1f}%')
        print(f'  {name:8s}  {cthr:>+8.0f}$  ' + '  '.join(cells))
        if name == 'POOLED':
            pooled_lifts = lifts
    return pooled_lifts

lift_1pct  = capture_table(s8,  fwd8,  0.01,  f'h{EXIT_H} score vs 2-min drops')
_          = capture_table(s8,  fwd8,  0.001, f'h{EXIT_H} score vs 2-min drops')
lift40     = capture_table(s40, fwd40, 0.01,  f'h{EXIT_H_ALT} score vs 10-min drops')

# tail monotonicity: mean forward move by score decile (pooled)
edges = np.quantile(s8, np.linspace(0.1, 0.9, 9))
dec   = np.digitize(s8, edges)
print(f'\n──── mean 2-min forward move (USD) by h{EXIT_H} score decile ────')
print('  ' + '  '.join(f'D{i}:{fwd8[dec == i].mean():+6.1f}' for i in range(10)))

# ── the gate (pre-registered, two-tier) ───────────────────────────────────
# Tier 1: broad tail — worst-1% 2-min drops. Much of that tail is ordinary
# vol noise, so even a perfect cascade detector may cap out below 2x here.
# Tier 2: deep tail — worst-0.1% drops, where cascade-type events dominate;
# a working detector must show a large lift there. Either tier passing means
# the score sees the tail; both failing kills the overlay.
def decile_lifts(fwd, crash_q):
    lifts = []
    for k in range(N_FOLDS):
        m = fold_of == k
        crash = fwd[m] <= np.quantile(fwd[m], crash_q)
        lifts.append((s8[m][crash] <= np.quantile(s8[m], 0.10)).mean() / 0.10)
    m = np.ones(len(fwd), dtype=bool)
    crash = fwd <= np.quantile(fwd, crash_q)
    pooled = (s8[crash] <= np.quantile(s8, 0.10)).mean() / 0.10
    return pooled, lifts

p1, f1 = decile_lifts(fwd8, 0.01)
p01, f01 = decile_lifts(fwd8, 0.001)
tier1 = (p1  >= 2.0) and (min(f1)  >= 1.5)
tier2 = (p01 >= 3.0) and (min(f01) >= 2.0)
GATE_PASS = tier1 or tier2
print('\nGATE (bottom-decile h8 score capture, pre-registered):')
print(f'  tier 1 — worst-1%   drops: pooled {p1:.2f}x (need ≥2.0), '
      f'worst fold {min(f1):.2f}x (need ≥1.5) → {"pass" if tier1 else "fail"}')
print(f'  tier 2 — worst-0.1% drops: pooled {p01:.2f}x (need ≥3.0), '
      f'worst fold {min(f01):.2f}x (need ≥2.0) → {"pass" if tier2 else "fail"}')
print(f'  → {"PASSED — overlay backtest below is meaningful" if GATE_PASS else "FAILED — the h8 score is (near-)blind to the tail; treat everything below as noise"}')


In [ ]:
# ── Cell 11: EMA crossover signal on 5-min bars ───────────────────────────
# 5-min closes from the 15s mid; EMA state is invalidated for one slow-span
# after any >15-min hole (ewm would otherwise carry a stale value across a
# collector outage as if no time had passed). The 5-min signal is mapped onto
# each 15s bar as the latest COMPLETED 5-min bucket (label == t is the bucket
# that closes exactly at t, so it is known at t — no lookahead), and goes
# stale if the latest bucket is older than EMA_STALE_MAX.

mid5  = pd.Series(close, index=pd.DatetimeIndex(dt_all)).resample(
            EMA_BAR, label='right', closed='right').last().dropna()
lab_s = mid5.index.values.astype('datetime64[s]').astype(np.int64)
bar5_gap = np.diff(lab_s, prepend=lab_s[0])
breaks5  = np.flatnonzero(bar5_gap > 900)          # >15-min hole in 5-min bars
print(f'5-min bars: {len(mid5):,}   holes >15min: {len(breaks5)}')

def ema_signal(fast, slow):
    ef  = mid5.ewm(span=fast, adjust=False).mean().values
    es  = mid5.ewm(span=slow, adjust=False).mean().values
    sig = np.sign(ef - es)
    ok  = np.ones(len(sig), dtype=bool)
    ok[:slow] = False                              # initial warm-up
    for b in breaks5:
        ok[b:b + slow] = False                     # re-converge after a hole
    sig[~ok] = 0.0
    return sig

t_pool = dt_s[ends_pool]
j      = np.searchsorted(lab_s, t_pool, side='right') - 1
stale  = (j < 0) | (t_pool - lab_s[np.maximum(j, 0)] > EMA_STALE_MAX)

ma_sig = {}
print(f"\n  {'pair':>8s}  {'long%':>6s}  {'short%':>7s}  {'flat%':>6s}  {'crosses':>8s}")
for pair in EMA_PAIRS:
    s5  = ema_signal(*pair)
    sig = s5[np.maximum(j, 0)].copy()
    sig[stale] = 0.0
    ma_sig[pair] = sig
    nz = sig[sig != 0.0]
    crosses = int((nz[1:] * nz[:-1] < 0).sum())
    print(f'  {pair[0]:>3d}/{pair[1]:<4d}  {np.mean(sig > 0)*100:>5.1f}%  '
          f'{np.mean(sig < 0)*100:>6.1f}%  {np.mean(sig == 0)*100:>5.1f}%  {crosses:>8,}')
print('\n(cross counts are over the pooled test period, '
      f'{pd.Timestamp(dt_all[ends_pool[0]])} → {pd.Timestamp(dt_all[ends_pool[-1]])})')


In [ ]:
# ── Cell 12: Overlay backtest — EMA entries, h8 fast exit ─────────────────
# State machine per bar: follow the MA signal; while in a position, force-exit
# when the exit score crosses −k·σ against it; after a fast exit stay flat
# (cooldown + hysteresis) until the score recovers above −k·RESUME_FRAC while
# the MA still points the same way; fresh MA entries are vetoed under the same
# condition. Fee attribution: MA-driven legs can be maker (planned), overlay
# legs are always taker (urgent by construction). "net real" = maker MA +
# taker overlay; "net taker" = everything taker (conservative bound).
# k = inf reproduces the pure-MA baseline exactly.

price   = close[ends_pool]
cont_ok = np.zeros(len(ends_pool), dtype=bool)
cont_ok[:-1] = np.diff(t_pool) <= HOLD_GAP_MAX     # may hold from bar i to i+1

def simulate(ma, score, k, long_only=True, resume=RESUME_FRAC,
             cooldown=REENTRY_COOLDOWN, veto=ENTRY_VETO, persist=EXIT_PERSIST):
    m = len(ma)
    pos     = np.zeros(m)
    turn_ma = np.zeros(m)          # $ turnover of MA-driven position changes
    turn_ov = np.zeros(m)          # $ turnover of overlay exits / re-entries
    n_exit = n_re = 0
    p, blocked, cd, streak = 0.0, 0.0, 0, 0
    finite_k = np.isfinite(k)
    for i in range(m):
        d = ma[i]
        if long_only and d < 0.0:
            d = 0.0
        if blocked != 0.0 and d != blocked:
            blocked = 0.0                          # MA left that direction
        if cd > 0:
            cd -= 1
        tgt, ov = d, False
        if p != 0.0 and d == p:
            # fast exit needs `persist` consecutive bars beyond −k: one-bar
            # spikes are mostly noise and every false alarm costs a round trip
            streak = streak + 1 if (finite_k and score[i] * p <= -k) else 0
            if streak >= persist:
                tgt, ov = 0.0, True
                blocked, cd, streak = p, cooldown, 0
                n_exit += 1
        else:
            streak = 0
            if p == 0.0 and d != 0.0:
                if blocked == d:                   # bailed out of / vetoed on d
                    if cd == 0 and score[i] * d > -k * resume:
                        tgt, ov = d, True          # score recovered → re-enter
                        n_re += 1
                        blocked = 0.0
                    else:
                        tgt = 0.0
                elif veto and finite_k and score[i] * d <= -k:
                    blocked, tgt = d, 0.0          # don't open into a crash signal
        if not cont_ok[i]:
            tgt, ov = 0.0, False                   # can't hold across a gap / last bar
        if tgt != p:
            (turn_ov if ov else turn_ma)[i] = abs(tgt - p) * price[i]
            p = tgt
        pos[i] = p
    pnl = pos[:-1] * np.diff(price)
    return {'pos': pos, 'pnl': pnl, 'turn_ma': turn_ma, 'turn_ov': turn_ov,
            'fees_real': turn_ma * MAKER_FEE + turn_ov * TAKER_FEE,
            'fees_cons': (turn_ma + turn_ov) * TAKER_FEE,
            'n_exit': n_exit, 'n_re': n_re}

def stats(bt):
    eq = np.cumsum(np.r_[0.0, bt['pnl']]) - np.cumsum(bt['fees_real'])
    dd = np.maximum.accumulate(eq) - eq
    ch   = np.flatnonzero(np.diff(np.r_[0.0, bt['pos']]) != 0)
    ends = np.r_[ch[1:], len(bt['pos'])]
    ep   = [(bt['pnl'][a:min(b, len(bt['pnl']))].sum(), b - a)
            for a, b in zip(ch, ends) if bt['pos'][a] != 0]
    tp    = np.array([x[0] for x in ep]) if ep else np.array([0.0])
    holds = np.array([x[1] for x in ep]) if ep else np.array([0])
    gross = bt['pnl'].sum()
    return {'eq': eq, 'maxdd': dd.max(), 'gross': gross,
            'net_real': gross - bt['fees_real'].sum(),
            'net_taker': gross - bt['fees_cons'].sum(),
            'n_trades': len(tp), 'win': (tp > 0).mean() * 100,
            'cover': (bt['pos'] != 0).mean() * 100,
            'hold_h': holds.mean() * WINDOW_SEC / 3600}

def overlay_table(pair, score, long_only=True, ks=EXIT_KS, label=f'h{EXIT_H}',
                  persist=EXIT_PERSIST):
    ma   = ma_sig[pair]
    base = simulate(ma, score, np.inf, long_only)
    sb   = stats(base)
    mode = 'long-only' if long_only else 'long/short'
    print(f'\n════ EMA {pair[0]}/{pair[1]} {mode} — fast exit on {label}, '
          f'persist={persist} ════')
    print(f"  base:  {sb['n_trades']:>5,} trades  cover {sb['cover']:>5.1f}%  "
          f"hold {sb['hold_h']:>5.1f}h  win {sb['win']:>4.1f}%  "
          f"gross {sb['gross']:>+9,.0f}  net(real) {sb['net_real']:>+9,.0f}  "
          f"net(tkr) {sb['net_taker']:>+9,.0f}  maxDD {sb['maxdd']:>9,.0f}")
    print(f"  {'k':>4s}  {'trades':>6s}  {'fastex':>6s}  {'re-en':>5s}  {'ovfee':>7s}  "
          f"{'win%':>5s}  {'gross':>9s}  {'net real':>9s}  {'net tkr':>9s}  "
          f"{'maxDD':>9s}  {'Δnet':>8s}  {'ΔDD':>8s}")
    out = {}
    for k in ks:
        bt = simulate(ma, score, k, long_only, persist=persist)
        s  = stats(bt)
        ovfee = bt['turn_ov'].sum() * TAKER_FEE
        dnet, ddd = s['net_real'] - sb['net_real'], s['maxdd'] - sb['maxdd']
        ok = '✓' if (dnet >= 0 and ddd <= -0.20 * sb['maxdd']) else ' '
        print(f"  {k:>4.1f}  {s['n_trades']:>6,}  {bt['n_exit']:>6,}  {bt['n_re']:>5,}  "
              f"{ovfee:>7,.0f}  {s['win']:>5.1f}  {s['gross']:>+9,.0f}  "
              f"{s['net_real']:>+9,.0f}  {s['net_taker']:>+9,.0f}  "
              f"{s['maxdd']:>9,.0f}  {dnet:>+8,.0f}  {ddd:>+8,.0f}  {ok}")
        out[k] = (bt, s)
    return base, sb, out

def per_fold_delta(pair, score, k, long_only=True):
    ma = ma_sig[pair]
    base, bt = simulate(ma, score, np.inf, long_only), simulate(ma, score, k, long_only)
    print(f'  per-fold Δnet(real) at k={k}: ', end='')
    for f in range(N_FOLDS):
        m  = fold_of == f
        mp = m[:-1]                                # pnl is per bar-transition
        d  = ((bt['pnl'][mp].sum()   - bt['fees_real'][m].sum()) -
              (base['pnl'][mp].sum() - base['fees_real'][m].sum()))
        print(f'  f{f}: {d:>+8,.0f}', end='')
    print()

# ── primary pair (pre-registered), h8 exits ───────────────────────────────
primary_base, primary_sb, primary_res = overlay_table(EMA_PRIMARY, s8, long_only=True)
for k in EXIT_KS:
    per_fold_delta(EMA_PRIMARY, s8, k, long_only=True)

# primary pair without the persistence filter — how much churn does the
# 2-consecutive-bars requirement save vs how much capture does it cost?
_ = overlay_table(EMA_PRIMARY, s8, long_only=True, persist=1)

# primary pair: h40 head as the exit signal (10-min horizon may match the
# speed of a developing collapse better than 2-min)
_ = overlay_table(EMA_PRIMARY, s40, long_only=True, label=f'h{EXIT_H_ALT}')

# primary pair, long/short variant
_ = overlay_table(EMA_PRIMARY, s8, long_only=False)

# ── exploratory grid (hypothesis-generating only) ─────────────────────────
for pair in EMA_PAIRS:
    if pair != EMA_PRIMARY:
        _ = overlay_table(pair, s8, long_only=True)

# ── random-exit control (primary pair): is the overlay better than exiting
# at random times with the same trigger statistics? Circular shifts preserve
# the score's distribution and autocorrelation but destroy alignment. ──────
rng_ctrl = np.random.default_rng(0)
DAY_BARS = 24 * 3600 // WINDOW_SEC
print(f'\n════ Random-exit control — EMA {EMA_PRIMARY[0]}/{EMA_PRIMARY[1]} long-only, '
      f'{N_CONTROL} circular shifts ════')
print(f"  {'k':>4s}  {'Δnet true':>9s}  {'Δnet ctrl (mean±sd)':>22s}  {'beats':>6s}   "
      f"{'ΔDD true':>9s}  {'ΔDD ctrl (mean±sd)':>22s}  {'beats':>6s}")
ma_p = ma_sig[EMA_PRIMARY]
for k in EXIT_KS:
    st  = primary_res[k][1]
    d_net_t = st['net_real'] - primary_sb['net_real']
    d_dd_t  = st['maxdd']    - primary_sb['maxdd']
    c_net, c_dd = [], []
    for _ in range(N_CONTROL):
        off = int(rng_ctrl.integers(DAY_BARS, len(s8) - DAY_BARS))
        sc  = stats(simulate(ma_p, np.roll(s8, off), k, long_only=True))
        c_net.append(sc['net_real'] - primary_sb['net_real'])
        c_dd.append(sc['maxdd'] - primary_sb['maxdd'])
    c_net, c_dd = np.array(c_net), np.array(c_dd)
    beats_net = (c_net < d_net_t).mean() * 100
    beats_dd  = (c_dd > d_dd_t).mean() * 100    # lower ΔDD is better
    print(f'  {k:>4.1f}  {d_net_t:>+9,.0f}  {c_net.mean():>+11,.0f} ±{c_net.std():>8,.0f}  '
          f'{beats_net:>5.0f}%   {d_dd_t:>+9,.0f}  {c_dd.mean():>+11,.0f} ±{c_dd.std():>8,.0f}  '
          f'{beats_dd:>5.0f}%')
print('\n"beats" = share of random controls the true overlay outperforms; the')
print('pre-registered bar is ≥95% on ΔDD. If controls score similar Δs, the')
print('overlay is just reducing market time, not detecting collapses.')


In [ ]:
# ── Cell 13: Plots ─────────────────────────────────────────────────────────
best_k  = max(EXIT_KS, key=lambda k: primary_res[k][1]['net_real'])
bt_best, st_best = primary_res[best_k]
dtp = dt_all[ends_pool]

fig, axes = plt.subplots(2, 1, figsize=(13, 9), height_ratios=[2, 1])

ax = axes[0]
ax.plot(dtp, price - price[0], color='0.6', lw=0.8, label='B&H (1 BTC)')
ax.plot(dtp, primary_sb['eq'], color='tab:blue', lw=1.0,
        label=f'EMA {EMA_PRIMARY[0]}/{EMA_PRIMARY[1]} long-only (net real fees)')
ax.plot(dtp, st_best['eq'], color='tab:red', lw=1.0,
        label=f'+ h{EXIT_H} fast exit, k={best_k} (net real fees)')
for f_ in folds:
    ax.axvline(dt_all[f_['test'][0]], color='k', lw=0.5, ls=':')
ax.set_title(f'Pooled walk-forward test — equity (USD, 1 BTC). '
             f'maxDD: base {primary_sb["maxdd"]:,.0f} → overlay {st_best["maxdd"]:,.0f}')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)

ax = axes[1]
means = [fwd8[dec == i].mean() for i in range(10)]
ax.bar(range(10), means, color=['tab:red'] * 2 + ['0.7'] * 8)
ax.set_xticks(range(10))
ax.set_xticklabels([f'D{i}' for i in range(10)])
ax.set_title(f'Mean 2-min forward move (USD) by h{EXIT_H} score decile — '
             'the overlay lives entirely in the red bars')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'run005_overlay.png'), dpi=120)
plt.show()


In [ ]:
# ── Cell 14: Save to Drive + how to read this run ─────────────────────────
import shutil

model_path = os.path.join(OUTPUT_DIR, f'lstm_run005_h{H_TRADE}.pt')
torch.save({
    'model_states': results[-1]['states'],     # all seeds, last (largest-train) fold
    'scaler':       results[-1]['scaler'],
    'features':     features,
    'horizons':     HORIZONS,
    'seeds':        SEEDS,
    'config':       dict(seq_len=SEQ_LEN, hidden_dim=HIDDEN_DIM,
                         num_layers=NUM_LAYERS, dropout=DROPOUT,
                         vol_window=VOL_WINDOW, target_clip=TARGET_CLIP,
                         window_sec=WINDOW_SEC,
                         sample_w_base=SAMPLE_W_BASE,
                         loss_w_alpha=LOSS_W_ALPHA, loss_w_cap=LOSS_W_CAP),
    'test_ic_per_fold':      [r['test_ic'] for r in results],
    'seed_test_ic_per_fold': [r['seed_test_ic'] for r in results],
}, model_path)
print(f'Saved model → {model_path}')

# run.004 lost its scores when the VM died — copy everything to Drive NOW
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
for p in [scores_path, model_path, os.path.join(OUTPUT_DIR, 'run005_overlay.png')]:
    if os.path.exists(p):
        shutil.copy(p, DRIVE_SAVE_DIR)
        print(f'  → Drive: {os.path.join(DRIVE_SAVE_DIR, os.path.basename(p))}')
drive.flush_and_unmount()

print('\nHow to read this run:')
print(' 1. Cell 10 GATE first. If the h8 score does not capture the worst-1%')
print('    2-min drops at ≥2x random (and ≥1.5x in every fold), STOP — every')
print('    overlay table below it is noise, whatever it says.')
print(' 2. Only the EMA 20/50 long-only table is pre-registered evidence. The')
print('    overlay wins if some k cuts maxDD ≥20% AND net(real) ≥ baseline AND')
print('    beats ≥95% of random controls on ΔDD, with neighbouring k agreeing.')
print(' 3. ovfee = fees burned by overlay churn (the entire cost of the idea);')
print('    fastex vs re-en shows how often it bails out vs how fast it gets back in.')
print(' 4. Other pairs / long-short / h40-exit tables are exploratory. If one of')
print('    them looks great while 20/50 fails, that is a hypothesis for run.006,')
print('    not a result.')
print(' 5. Slippage is not modelled: taker fills during a cascade are worse than')
print('    the bar close used here — treat positive results as an upper bound.')
